## Installation

In [1]:
!pip install sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 111.6 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitli

In [2]:
!pip install pytrec_eval

  Preparing metadata (setup.py) ... done
  Created wheel for pytrec_eval: filename=pytrec_eval-0.5-cp311-cp311-linux_x86_64.whl size=308668 sha256=65f2d1e15dfa3f95191205b4341810fc2bf1d7b7bb1b7a30de0b46a55a972dd4
  Stored in directory: /root/.cache/pip/wheels/0f/89/42/86aecdb99975f1840c27bc37fdfed72116abcf82e2c9dc76a8
Successfully built pytrec_eval


### Imports

In [3]:
"""
This examples show how to train a Cross-Encoder for the MS Marco dataset (https://github.com/microsoft/MSMARCO-Passage-Ranking).

The query and the passage are passed simoultanously to a Transformer network. The network then returns
a score between 0 and 1 how relevant the passage is for a given query.

The resulting Cross-Encoder can then be used for passage re-ranking: You retrieve for example 100 passages
for a given query, for example with ElasticSearch, and pass the query+retrieved_passage to the CrossEncoder
for scoring. You sort the results then according to the output of the CrossEncoder.

This gives a significant boost compared to out-of-the-box ElasticSearch / BM25 ranking.
"""
from torch.utils.data import DataLoader
from sentence_transformers import LoggingHandler, util
from sentence_transformers.cross_encoder import CrossEncoder
from sentence_transformers.cross_encoder.evaluation import CERerankingEvaluator
from sentence_transformers import InputExample
from datetime import datetime
import gzip
import os
import tarfile
import tqdm
import logging
from collections import defaultdict
import numpy as np
import sys
import pytrec_eval
logger = logging.getLogger()
logger.setLevel(logging.DEBUG)
logging.basicConfig(format='%(asctime)s - %(message)s',datefmt='%Y-%m-%d %H:%M:%S')

## Evaluating preparation

### Initialize hyperparameters (e.g., batch size, etc)

In [4]:
from google.colab import drive
drive.mount('/content/gdrive')
base_path = "./gdrive/MyDrive/cross-encoder-reranker-ir-course-2023/"

Mounted at /content/gdrive


In [5]:
!mkdir -p $base_path

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Evaluate the models


### Load the fine-tuned cross encoder ms marco MiniLM model

---



In [16]:
minilm_model_save_path = "/content/drive/MyDrive/cross-encoder-reranker-ir-course-2023/finetuned_models/cross-encoder-ms-marco-MiniLM-L-2-v2" #@param {type:"string"}

### Load the fine-tuned cross encoder ms marco distilroberta

In [18]:
distilroberta_model_save_path = "/content/drive/MyDrive/cross-encoder-reranker-ir-course-2023/finetuned_models/cross-encoder-distilroberta-base" #@param {type:"string"}

### Load the fine-tuned cross encoder tinybert

In [19]:
tinybert_model_save_path = "/content/drive/MyDrive/cross-encoder-reranker-ir-course-2023/finetuned_models/cross-encoder-ms-marco-tinybert-l-2-v2" #@param {type:"string"}

### Load data (For evaluation on TREC DL'19)

In [10]:
!wget https://msmarco.z22.web.core.windows.net/msmarcoranking/queries.tar.gz
!tar -xvzf  queries.tar.gz

--2025-05-09 10:52:55--  https://msmarco.z22.web.core.windows.net/msmarcoranking/queries.tar.gz
Resolving msmarco.z22.web.core.windows.net (msmarco.z22.web.core.windows.net)... 20.150.34.1
Connecting to msmarco.z22.web.core.windows.net (msmarco.z22.web.core.windows.net)|20.150.34.1|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 18882551 (18M) [application/gzip]
Saving to: ‘queries.tar.gz’

queries.tar.gz      100%[===================>]  18.01M  5.24MB/s    in 4.1s    

2025-05-09 10:53:00 (4.41 MB/s) - ‘queries.tar.gz’ saved [18882551/18882551]

queries.dev.tsv
queries.eval.tsv
queries.train.tsv


In [11]:
"""
This file evaluates CrossEncoder on the TREC 2019 Deep Learning (DL) Track: https://arxiv.org/abs/2003.07820

TREC 2019 DL is based on the corpus of MS Marco. MS Marco provides a sparse annotation, i.e., usually only a single
passage is marked as relevant for a given query. Many other highly relevant passages are not annotated and hence are treated
as an error if a model ranks those high.

TREC DL instead annotated up to 200 passages per query for their relevance to a given query. It is better suited to estimate
the model performance for the task of reranking in Information Retrieval.

Run:
python eval_cross-encoder-trec-dl.py cross-encoder-model-name

"""


data_folder = 'trec2019-data'
os.makedirs(data_folder, exist_ok=True)

#Read test queries
queries = {}
queries_filepath = os.path.join(data_folder, 'msmarco-test2019-queries.tsv.gz')
if not os.path.exists(queries_filepath):
    logging.info("Download "+os.path.basename(queries_filepath))
    util.http_get('https://msmarco.z22.web.core.windows.net/msmarcoranking/msmarco-test2019-queries.tsv.gz', queries_filepath)

with gzip.open(queries_filepath, 'rt', encoding='utf8') as fIn:
    for line in fIn:
        qid, query = line.strip().split("\t")
        queries[qid] = query

#Read which passages are relevant
relevant_docs = defaultdict(lambda: defaultdict(int))
qrels_filepath = os.path.join(data_folder, '2019qrels-pass.txt')

if not os.path.exists(qrels_filepath):
    logging.info("Download "+os.path.basename(qrels_filepath))
    util.http_get('https://trec.nist.gov/data/deep/2019qrels-pass.txt', qrels_filepath)


with open(qrels_filepath) as fIn:
    for line in fIn:
        qid, _, pid, score = line.strip().split()
        score = int(score)
        if score > 0:
            relevant_docs[qid][pid] = score

# Only use queries that have at least one relevant passage
relevant_qid = []
for qid in queries:
    if len(relevant_docs[qid]) > 0:
        relevant_qid.append(qid)


# Read the top 1000 passages that are supposed to be re-ranked
passage_filepath = os.path.join(data_folder, 'msmarco-passagetest2019-top1000.tsv.gz')

if not os.path.exists(passage_filepath):
    logging.info("Download "+os.path.basename(passage_filepath))
    util.http_get('https://msmarco.z22.web.core.windows.net/msmarcoranking/msmarco-passagetest2019-top1000.tsv.gz', passage_filepath)



passage_cand = {}
with gzip.open(passage_filepath, 'rt', encoding='utf8') as fIn:
    for line in fIn:
        qid, pid, query, passage = line.strip().split("\t")
        if qid not in passage_cand:
            passage_cand[qid] = []

        passage_cand[qid].append([pid, passage])

logging.info("Queries: {}".format(len(queries)))


INFO:root:Download msmarco-test2019-queries.tsv.gz
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): msmarco.z22.web.core.windows.net:443
DEBUG:urllib3.connectionpool:https://msmarco.z22.web.core.windows.net:443 "GET /msmarcoranking/msmarco-test2019-queries.tsv.gz HTTP/1.1" 200 4276


  0%|          | 0.00/4.28k [00:00<?, ?B/s]

INFO:root:Download 2019qrels-pass.txt
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): trec.nist.gov:443
DEBUG:urllib3.connectionpool:https://trec.nist.gov:443 "GET /data/deep/2019qrels-pass.txt HTTP/1.1" 200 None


0.00B [00:00, ?B/s]

INFO:root:Download msmarco-passagetest2019-top1000.tsv.gz
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): msmarco.z22.web.core.windows.net:443
DEBUG:urllib3.connectionpool:https://msmarco.z22.web.core.windows.net:443 "GET /msmarcoranking/msmarco-passagetest2019-top1000.tsv.gz HTTP/1.1" 200 26634062


  0%|          | 0.00/26.6M [00:00<?, ?B/s]

INFO:root:Queries: 200


# Predictions

## MiniLM Predictions

In [17]:
minilm_queries_result_list = []
minilm_run = {}
minilm_model = CrossEncoder(minilm_model_save_path, max_length=512)

for qid in tqdm.tqdm(relevant_qid):
    query = queries[qid]

    cand = passage_cand[qid]
    pids = [c[0] for c in cand]
    corpus_sentences = [c[1] for c in cand]

    cross_inp = [[query, sent] for sent in corpus_sentences]

    if minilm_model.config.num_labels > 1: #Cross-Encoder that predict more than 1 score, we use the last and apply softmax
        cross_scores = minilm_model.predict(cross_inp, apply_softmax=True)[:, 1].tolist()
    else:
        cross_scores = minilm_model.predict(cross_inp).tolist()

    cross_scores_sparse = {}
    for idx, pid in enumerate(pids):
        cross_scores_sparse[pid] = cross_scores[idx]

    sparse_scores = cross_scores_sparse
    minilm_run[qid] = {}
    for pid in sparse_scores:
        minilm_run[qid][pid] = float(sparse_scores[pid])

INFO:sentence_transformers.cross_encoder.CrossEncoder:Use pytorch device: cuda
  0%|          | 0/43 [00:00<?, ?it/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

  2%|▏         | 1/43 [00:01<00:59,  1.41s/it]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

  5%|▍         | 2/43 [00:02<00:52,  1.29s/it]

...

 95%|█████████▌| 41/43 [00:27<00:00,  2.08it/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

 98%|█████████▊| 42/43 [00:28<00:00,  2.04it/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

100%|██████████| 43/43 [00:28<00:00,  1.49it/s]


## Distilroberta Predictions

In [20]:
distilroberta_queries_result_list = []
distilroberta_run = {}
distilroberta_model = CrossEncoder(distilroberta_model_save_path, max_length=512)

for qid in tqdm.tqdm(relevant_qid):
    query = queries[qid]

    cand = passage_cand[qid]
    pids = [c[0] for c in cand]
    corpus_sentences = [c[1] for c in cand]

    cross_inp = [[query, sent] for sent in corpus_sentences]

    if distilroberta_model.config.num_labels > 1: #Cross-Encoder that predict more than 1 score, we use the last and apply softmax
        cross_scores = distilroberta_model.predict(cross_inp, apply_softmax=True)[:, 1].tolist()
    else:
        cross_scores = distilroberta_model.predict(cross_inp).tolist()

    cross_scores_sparse = {}
    for idx, pid in enumerate(pids):
        cross_scores_sparse[pid] = cross_scores[idx]

    sparse_scores = cross_scores_sparse
    distilroberta_run[qid] = {}
    for pid in sparse_scores:
        distilroberta_run[qid][pid] = float(sparse_scores[pid])

INFO:sentence_transformers.cross_encoder.CrossEncoder:Use pytorch device: cuda
  0%|          | 0/43 [00:00<?, ?it/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

  2%|▏         | 1/43 [00:03<02:45,  3.95s/it]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

  5%|▍         | 2/43 [00:07<02:42,  3.95s/it]

...

 95%|█████████▌| 41/43 [02:39<00:07,  3.83s/it]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

 98%|█████████▊| 42/43 [02:43<00:03,  3.90s/it]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

100%|██████████| 43/43 [02:48<00:00,  3.91s/it]


## Tinybert Predictions

In [21]:
tinybert_queries_result_list = []
tinybert_run = {}
tinybert_model = CrossEncoder(tinybert_model_save_path, max_length=512)

for qid in tqdm.tqdm(relevant_qid):
    query = queries[qid]

    cand = passage_cand[qid]
    pids = [c[0] for c in cand]
    corpus_sentences = [c[1] for c in cand]

    cross_inp = [[query, sent] for sent in corpus_sentences]

    if tinybert_model.config.num_labels > 1: #Cross-Encoder that predict more than 1 score, we use the last and apply softmax
        cross_scores = tinybert_model.predict(cross_inp, apply_softmax=True)[:, 1].tolist()
    else:
        cross_scores = tinybert_model.predict(cross_inp).tolist()

    cross_scores_sparse = {}
    for idx, pid in enumerate(pids):
        cross_scores_sparse[pid] = cross_scores[idx]

    sparse_scores = cross_scores_sparse
    tinybert_run[qid] = {}
    for pid in sparse_scores:
        tinybert_run[qid][pid] = float(sparse_scores[pid])

INFO:sentence_transformers.cross_encoder.CrossEncoder:Use pytorch device: cuda
  0%|          | 0/43 [00:00<?, ?it/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

  2%|▏         | 1/43 [00:00<00:20,  2.02it/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

  5%|▍         | 2/43 [00:01<00:20,  1.99it/s]

...

 95%|█████████▌| 41/43 [00:26<00:01,  1.44it/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

 98%|█████████▊| 42/43 [00:26<00:00,  1.35it/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

100%|██████████| 43/43 [00:27<00:00,  1.55it/s]


# Evaluation

In [22]:
evaluator = pytrec_eval.RelevanceEvaluator(relevant_docs, {'ndcg_cut.10', 'recall_100', 'map_cut.1000'})

## MiniLM Evaluation

In [23]:
minilm_scores = evaluator.evaluate(minilm_run)

print("Queries:", len(relevant_qid))
print("NDCG@10: {:.2f}".format(np.mean([ele["ndcg_cut_10"] for ele in minilm_scores.values()])*100))
print("Recall@100: {:.2f}".format(np.mean([ele["recall_100"] for ele in minilm_scores.values()])*100))
print("MAP@1000: {:.2f}".format(np.mean([ele["map_cut_1000"] for ele in minilm_scores.values()])*100))

Queries: 43
NDCG@10: 68.99
Recall@100: 50.68
MAP@1000: 44.88


## Distilroberta Evaluation

In [24]:
distilroberta_scores = evaluator.evaluate(distilroberta_run)

print("Queries:", len(relevant_qid))
print("NDCG@10: {:.2f}".format(np.mean([ele["ndcg_cut_10"] for ele in distilroberta_scores.values()])*100))
print("Recall@100: {:.2f}".format(np.mean([ele["recall_100"] for ele in distilroberta_scores.values()])*100))
print("MAP@1000: {:.2f}".format(np.mean([ele["map_cut_1000"] for ele in distilroberta_scores.values()])*100))

Queries: 43
NDCG@10: 63.10
Recall@100: 47.95
MAP@1000: 40.74


## Tinybert Evaluation

In [26]:
tinybert_scores = evaluator.evaluate(tinybert_run)

print("Queries:", len(relevant_qid))
print("NDCG@10: {:.2f}".format(np.mean([ele["ndcg_cut_10"] for ele in tinybert_scores.values()])*100))
print("Recall@100: {:.2f}".format(np.mean([ele["recall_100"] for ele in tinybert_scores.values()])*100))
print("MAP@1000: {:.2f}".format(np.mean([ele["map_cut_1000"] for ele in tinybert_scores.values()])*100))

Queries: 43
NDCG@10: 69.50
Recall@100: 50.49
MAP@1000: 45.68


## Sorting candidate documents of each query based on their relevance score

## MiniLM Sorting

In [27]:
import operator
for qid in minilm_run.keys():
  minilm_run[qid] = sorted(minilm_run[qid].items(), key=operator.itemgetter(1), reverse = True)

## Distilroberta Sorting

In [28]:
import operator
for qid in distilroberta_run.keys():
  distilroberta_run[qid] = sorted(distilroberta_run[qid].items(), key=operator.itemgetter(1), reverse = True)

## Tinybert Sorting

In [29]:
import operator
for qid in tinybert_run.keys():
  tinybert_run[qid] = sorted(tinybert_run[qid].items(), key=operator.itemgetter(1), reverse = True)

## Storing ranking run file

## MiniLM Storing Ranking Run File



In [31]:
minilm_ranking_lines = []
for qid in minilm_run.keys():
  for rank, did_pred_score in enumerate(minilm_run[qid]):
    did, pred_score = did_pred_score
    line = "{qid} Q0 {did} {rank} {pred_score} STANDARD".format(qid=qid, did=did, rank=rank, pred_score=str(pred_score))
    minilm_ranking_lines.append(line)

minilm_ranking_run_file_path = minilm_model_save_path + "ranking.run"
f_w = open(minilm_ranking_run_file_path, "w+")
f_w.write("\n".join(minilm_ranking_lines))
f_w.close()

### Print the first three lines of the stored ranking run file (MiniLM)

In [32]:
!head -n 3 $minilm_ranking_run_file_path

156493 Q0 1960255 0 7.4322614669799805 STANDARD
156493 Q0 1960260 1 7.086747169494629 STANDARD
156493 Q0 2928707 2 6.595431804656982 STANDARD


## Distilroberta Storing Ranking Run File


In [33]:
ranking_lines = []
for qid in distilroberta_run.keys():
  for rank, did_pred_score in enumerate(distilroberta_run[qid]):
    did, pred_score = did_pred_score
    line = "{qid} Q0 {did} {rank} {pred_score} STANDARD".format(qid=qid, did=did, rank=rank, pred_score=str(pred_score))
    ranking_lines.append(line)

distilroberta_ranking_run_file_path = distilroberta_model_save_path + "ranking.run"
f_w = open(distilroberta_ranking_run_file_path, "w+")
f_w.write("\n".join(ranking_lines))
f_w.close()

### Print the first three lines of the stored ranking run file (Distilroberta)

In [34]:
!head -n 3 $distilroberta_ranking_run_file_path

156493 Q0 1960255 0 0.9812994599342346 STANDARD
156493 Q0 1960260 1 0.9735860228538513 STANDARD
156493 Q0 8182162 2 0.9699329733848572 STANDARD


## Tinybert Storing Ranking Run File


In [35]:
ranking_lines = []
for qid in tinybert_run.keys():
  for rank, did_pred_score in enumerate(tinybert_run[qid]):
    did, pred_score = did_pred_score
    line = "{qid} Q0 {did} {rank} {pred_score} STANDARD".format(qid=qid, did=did, rank=rank, pred_score=str(pred_score))
    ranking_lines.append(line)

tinybert_ranking_run_file_path = tinybert_model_save_path + "ranking.run"
f_w = open(tinybert_ranking_run_file_path, "w+")
f_w.write("\n".join(ranking_lines))
f_w.close()

### Print the first three lines of the stored ranking run file (Tinybert)

In [36]:
!head -n 3 $ranking_run_file_path

156493 Q0 1960255 0 7.4322614669799805 STANDARD
156493 Q0 1960260 1 7.086747169494629 STANDARD
156493 Q0 2928707 2 6.595431804656982 STANDARD
